# 2. Join the federated-learning round

## Goal

This notebook starts the Flower client for **your group only**.

The organiser starts the shared aggregation server. Each group then trains
locally on its own partition and sends model updates—not its CSV rows—to
that server.

**Run the final cell only when the organiser asks all groups to connect.**


## Before you start

Confirm all three statements:

- You completed `01_Inspect_Local_Partition.ipynb`.
- The organiser has confirmed that the Flower server is ready.
- Your group will run the final cell once, and leave it running until it
  reports completion.

The final cell remains busy while the federated rounds are in progress.
This is expected.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import flwr

group_id = os.environ["GROUP_ID"]
workshop_root = Path(os.environ["DIGITAFRICA_WORKSHOP_ROOT"])
data_path = Path(os.environ["CLIENT_DATA_PATH"])
client_program = workshop_root / "app" / "client" / "client.py"
app_root = workshop_root / "app"

if not group_id.startswith("group_"):
    raise RuntimeError(
        f"This notebook requires a workshop group login, got {group_id!r}."
    )
if not data_path.is_file():
    raise FileNotFoundError(f"Local partition is unavailable: {data_path}")
if not client_program.is_file():
    raise FileNotFoundError(f"Workshop client is unavailable: {client_program}")

print("Client readiness check passed.")
print(f"Your group:       {group_id}")
print(f"Local partition: {data_path.name}")
print(f"Flower endpoint: {os.environ['FLOWER_SERVER_ADDRESS']}")
print(f"Flower version:  {flwr.__version__}")


## What happens in each federated-learning round?

The workshop server uses Flower’s **FedAvg** (federated averaging) strategy.
It waits until the configured groups have connected—normally two groups in the
current pilot—then coordinates five rounds by default.

In every round:

1. **Receive:** the server sends the current shared model to each group. The
   model consists of a numeric weights matrix and a bias vector.
2. **Train locally:** each group performs one local gradient-descent epoch by
   default, using only its own partition. The CSV identifiers and labels create
   deterministic **16-dimensional synthetic features**; no cassava image file
   is opened or processed.
3. **Return an update:** each group sends its updated model parameters, its
   local sample count, and local summary metrics. It does not send its CSV
   rows, labels, synthetic-feature matrix, or image files.
4. **Aggregate:** the server calculates the next shared model by averaging
   groups’ parameter updates, weighted by their local sample counts.

A group with more local samples therefore contributes proportionally more to
the shared parameter average. This is the central idea of FedAvg.


## What stays local, and what is shared?

During the workshop, your group keeps its local partition in its own workspace.

| Remains local to your group | Sent to the shared Flower server |
|---|---|
| CSV metadata rows | A model update: changed model parameters after local training |
| Image references | Summary information needed to coordinate the round |
| Local labels and synthetic feature matrix | No raw CSV file or image file |
| Detailed local training process | The contribution required to build the next shared model |

A **model update** is not a copy of the data. It represents how local training
changed the model's numeric parameters.

However, model updates can still reveal information in some circumstances.
This introductory demonstration does not implement advanced protections such
as secure aggregation or differential privacy. Federated learning reduces the
need to centralise raw data; it does not automatically solve every privacy,
security, or governance problem.


In [ ]:
if str(app_root) not in sys.path:
    sys.path.insert(0, str(app_root))

from client.client import load_partition

features, labels = load_partition(
    data_path,
    num_classes=5,
    feature_dim=16,
    seed=42,
)

print(
    f"{group_id} is ready to contribute {len(labels)} local examples "
    f"with {features.shape[1]} synthetic features each."
)
print("No local CSV rows will be sent to the Flower server.")


## Start the client

When the organiser gives the instruction, run the next cell **once**.

You will see connection and training output below the cell. Keep the cell
running. If it reports an error, do not repeatedly restart it; notify the
organiser and include the visible message.


## How to read the client output and evaluate the demonstration

The final cell first prints a line beginning **`Starting workflow-demo client:`**.
Check that it shows your expected group identifier, local CSV path, sample
count, five classes, and 16 synthetic features.

Flower then logs connection, parameter exchange, training, evaluation, and
completion. The exact wording of Flower messages depends on the installed
Flower version. In addition, this workshop client explicitly prints:

- **`Local training complete:`** after each local training step;
- **`Local evaluation of shared model:`** after each evaluation of the shared
  model.

Each of these summaries identifies your group and shows its local sample
count, loss, and accuracy. They do not reveal CSV rows, image references,
labels, synthetic features, or model-parameter values.

A normal run follows this sequence:

1. **Connect:** your group connects to the organiser-controlled server.
2. **Receive:** the server provides the current shared model.
3. **Train locally:** your group completes one local training epoch on its own
   synthetic features and labels.
4. **Send update:** your group returns the updated weights and bias, together
   with its local sample count.
5. **Evaluate:** after aggregation, your group evaluates the shared model on
   its own local partition.
6. **Repeat and complete:** this continues for five rounds by default, after
   which the client exits successfully.

### What the metrics mean

The local client calculates:

- **loss:** cross-entropy loss on its own local partition; lower is generally
  better;
- **accuracy:** the proportion of local labels predicted correctly by the
  shared model.

The server combines the clients’ **evaluation loss**, weighting each result by
the number of local samples. In the current implementation, it does **not**
configure aggregation of the client accuracy metrics. Therefore:

- a local accuracy visible in one client’s output is not a global accuracy;
- the aggregated loss is evaluated on the participating groups’ training
  partitions, not on an independent held-out test set;
- neither metric validates a cassava disease classifier.

For this workshop, a successful result is that every assigned group connects,
all configured rounds finish, and the client exits without an error. The
metrics illustrate the workflow; they are not evidence of real-world model
quality.

If the cell remains busy during the rounds, that is expected. Do not run it a
second time unless the organiser specifically instructs you to do so.


In [ ]:
client_environment = os.environ.copy()
client_environment["PYTHONUNBUFFERED"] = "1"

print(f"Starting the Flower client for {group_id}...")
print("Waiting for the organiser-controlled server and other groups...\n")

process = subprocess.Popen(
    [sys.executable, "-u", str(client_program)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=client_environment,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

exit_code = process.wait()
if exit_code != 0:
    raise RuntimeError(
        f"Flower client for {group_id} stopped with exit code {exit_code}. "
        "Tell the organiser and include the output above."
    )

print(f"\nFederated-learning client for {group_id} completed successfully.")


## Optional follow-up: learn more about Flower

You do **not** need this resource to run the client in this workshop. It may be
useful before or after the session if you would like a broader technical
introduction to the framework used here.

**[Introduction to Federated Learning with Flower — A Hands-on Tutorial](https://www.classcentral.com/course/youtube-an-intro-to-federated-learning-with-flower-with-daniel-j-beutel-338865)**  
Presented by Flower co-creator Daniel J. Beutel — approximately 31 minutes.
It introduces Flower and demonstrates a first “Hello, Flower” workflow.

The workshop’s own client setup remains deliberately simpler: it is a guided,
controlled demonstration of local training and server aggregation.


## After completion

Discuss with the organiser:

- Which information remained local to each group?
- Which model information was shared with the server?
- Why does federated learning still require coordination, validation, and
  an agreed data-governance process?

This workshop is a workflow demonstration, not a validated cassava
disease-classification model.

**Additional reflection:** Why would a high score in this synthetic exercise not be enough evidence to deploy a cassava disease classifier?
